# Toy Example

- This notebook demonstrates how to run the code on your own data.
- In the synthetic data experiments, data can be loaded internally in `run_feature_selection_model` (no need to call `generate_data` manually).
- Multiple runs across different seeds, datasets and models can be done with `utils/run_multiple_tests.py`.

In [10]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
import repo_paths  # noqa: F401

import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2
from tools import run_feature_selection_model

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
seed = 0

In [12]:
from Data_Generation import generate_data

x_train, y_train, _ = generate_data(n=10_000, data_type='Syn4', seed=seed, out='Y', num_features=11)
x_test,  y_test,  g_test  = generate_data(n=10_000, data_type='Syn4', seed=seed+1, out='Y', num_features=11)

print('x_train shape:', x_train.shape)
print('y_train shape:', y_train.shape)
print('g_test shape: ', g_test.shape)

x_train shape: (10000, 11)
y_train shape: (10000, 2)
g_test shape:  (10000, 11)


In [13]:
full_data_dict = {
    'x_train': x_train,
    'y_train': y_train,
    'x_test':  x_test,
    'y_test':  y_test,
    'g_test':  g_test, #set to None if ground-truth feature importance is unknown. In that case, ground-truth metrics won't be calculated
}

In [15]:
results = run_feature_selection_model(
    full_data_dict=full_data_dict,
    model_type='hide_and_seek', #other models from the paper are available. see readme.txt for environment details
    lmbda=0.3, #should be tuned. see experiments/tests_synthetic/tuning_lmbda_synthetic.ipynb for details
    perturbation_method='draw_marginal',
    seed=seed,
    num_important_features=None, #not used in hide_and_seek, invase, realx (see paper). For other models, it is needed to calculate ground truth metrics - either specify an integer number of features or 'use_gtruth'.
    include_model=True,
    folder_for_pickle=None, #save results location
    scale_data=True #standardizes using train data
)

m=hide_and_seek_e=500_l=0.3_b=None_seed=0_k=None_f=11_bn=False_p=2_vl=False_dm=synthetic_en=None_cs=None_pm=mrgl_rho=0.0_sq=None_t=mc
scaling data
training model
experiment | Epoch: 0 | Loss: 0.6914
experiment | Epoch: 100 | Loss: 0.6209
experiment | Epoch: 200 | Loss: 0.5843
experiment | Epoch: 300 | Loss: 0.5758
experiment | Epoch: 400 | Loss: 0.5927
experiment | Epoch: 499 | Loss: 0.6229
training finished
experiment: TPR mean: 98.6%
experiment: FDR mean: 2.9%
experiment: F1 mean: 97.6%
experiment: pct_sig: 0.3739
experiment: roc_auc: 0.8068


- Further customizability can be seen in utils.tools.run_feature_selection_model (for example, ensembling and column subsampling)
- Network architecture (epochs, network depth and width, etc) has been validated with experiments. It is not recommended to change these values without first validating the approach with ground truth data

In [16]:
keys = ['TPR_mean', 'FDR_mean', 'f1', # instance-wise feature importance metrics - calculated when ground truth importance (g_test) is known
        'accuracy', 'roc_auc', 'pr_auc', # prediction metrics
        'binary_mask', 'mask', 'pct_sig', # instance-wise feature importance assigned by the model
        'model_type', 'model', # model information
        'perturbation_method', 'lmbda', 'seed', 'scale_data', 'num_important_features', # run settings (reminder num_important_features is not used by hide_and_seek)
        'g_test', 'y_test_pred', 'y_test', # data
        'num_classes', 'num_syn_features', 'train_N', 'test_N' # data summary
       ]

results = {k: results[k] for k in keys if k in results}

In [17]:
results

{'TPR_mean': 98.56533307170488,
 'FDR_mean': 2.9434761844857005,
 'f1': 97.57437085008988,
 'accuracy': 0.7179,
 'roc_auc': 0.8067791532291153,
 'pr_auc': 0.8227518171110726,
 'binary_mask': array([[0., 0., 1., ..., 0., 0., 1.],
        [0., 0., 1., ..., 0., 0., 1.],
        [1., 1., 0., ..., 0., 0., 1.],
        ...,
        [1., 1., 0., ..., 0., 0., 1.],
        [0., 0., 1., ..., 0., 0., 1.],
        [1., 1., 0., ..., 0., 0., 1.]]),
 'mask': array([[8.4627311e-07, 1.6631617e-06, 9.9999905e-01, ..., 7.0127471e-06,
         3.8795470e-06, 9.9999678e-01],
        [7.5838485e-05, 1.0352644e-04, 9.9991167e-01, ..., 1.5873733e-05,
         1.0508296e-05, 9.9999607e-01],
        [9.9909568e-01, 9.9883538e-01, 1.9899141e-03, ..., 1.8928924e-03,
         1.7287048e-03, 8.9355314e-01],
        ...,
        [9.9948585e-01, 9.9954396e-01, 1.1765513e-03, ..., 1.4126275e-03,
         7.3039922e-04, 8.0515134e-01],
        [3.0849500e-05, 4.6799774e-05, 9.9996889e-01, ..., 2.2286150e-04,
         7